# Milestone 2 · Build the Destiny lore corpus

This notebook builds the corpus the RAG system answers from. It pulls from three sources in three formats:

| Source | Format | What |
|---|---|---|
| Bungie.net API, Destiny 2 manifest | JSON | Lore entries, plus the lore book each one belongs to |
| Bungie.net API, Destiny 1 Grimoire | CSV (text contains HTML) | Grimoire cards, grouped by theme and page |
| Destiny Wiki (destiny.fandom.com) | HTML | Character, faction, and event pages, plus gameplay pages as distractors |

**Outputs**
- `data/corpus/docs.jsonl`: one cleaned document per line, the input to indexing. Git-ignored, because Bungie's text is copyrighted.
- `data/corpus/raw/`: raw downloads, cached so reruns skip the network. Git-ignored.
- `corpus/manifest.jsonl`: one line per document with source, format, license note, game, book and role. **Committed.**
- `corpus/corpus_report.md`: counts against the Corpus Sprint targets. **Committed.**

In [1]:
# Setup
import os, re, json, time, html, hashlib, numbers
from pathlib import Path
from getpass import getpass

import requests
import pandas as pd
import tiktoken
from bs4 import BeautifulSoup
from dotenv import load_dotenv

REPO = Path.cwd()
RAW = REPO / 'data' / 'corpus' / 'raw'
WIKI_RAW = RAW / 'wiki'
DOCS_PATH = REPO / 'data' / 'corpus' / 'docs.jsonl'
META_DIR = REPO / 'corpus'
for d in (RAW, WIKI_RAW, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

MIN_WORDS = 20          # drop entries shorter than this (mostly fragments)
WIKI_SLEEP = 1.0        # seconds between wiki API calls
USER_AGENT = 'DATA790-student-RAG-project/1.0 (educational use; github.com/henryhal)'

load_dotenv()
BUNGIE_API_KEY = os.environ.get('BUNGIE_API_KEY') or getpass('Paste your Bungie API key: ')
print('Bungie key loaded:', 'yes' if BUNGIE_API_KEY else 'no')

enc = tiktoken.get_encoding('cl100k_base')
def n_tokens(text):
    return len(enc.encode(text))

def norm_hash(text):
    # Hash of the text with case and whitespace normalized, used for dedup
    return hashlib.sha1(re.sub(r'\s+', ' ', text).strip().lower().encode()).hexdigest()

def slug(s):
    return re.sub(r'[^a-z0-9]+', '-', s.lower()).strip('-')[:60] or 'untitled'

/Users/henryhalvorsen/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Bungie key loaded: yes


## 1. Destiny 2 lore from the Bungie API (JSON)

Downloads three tables from the current manifest:
- `DestinyLoreDefinition`: the lore text
- `DestinyRecordDefinition`: records (lore pages) that point to a lore entry
- `DestinyPresentationNodeDefinition`: the lore books and shelves that hold those records

Each raw table is cached in `data/corpus/raw/`.

In [2]:
BUNGIE = 'https://www.bungie.net'
TABLES = ['DestinyLoreDefinition', 'DestinyRecordDefinition', 'DestinyPresentationNodeDefinition']

resp = requests.get(f'{BUNGIE}/Platform/Destiny2/Manifest/',
                    headers={'X-API-Key': BUNGIE_API_KEY}, timeout=30)
resp.raise_for_status()
manifest = resp.json()['Response']
MANIFEST_VERSION = manifest['version']
print('Manifest version:', MANIFEST_VERSION)

def get_table(name):
    path = RAW / f'd2_{name}.json'
    if not path.exists():
        url = BUNGIE + manifest['jsonWorldComponentContentPaths']['en'][name]
        print('Downloading', name, '...')
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.write_bytes(r.content)
    return json.loads(path.read_text())

lore_raw, records_raw, nodes_raw = (get_table(t) for t in TABLES)
print(f'lore: {len(lore_raw):,} | records: {len(records_raw):,} | presentation nodes: {len(nodes_raw):,}')

Manifest version: 245114.26.10.13.2000-1-bnet.66137
lore: 3,514 | records: 6,168 | presentation nodes: 2,375


In [3]:
# Map each lore entry to its book (the presentation node holding its record) and shelf (that node's parent)
record_to_node, node_parent = {}, {}
for nh, node in nodes_raw.items():
    children = node.get('children', {})
    for rec in children.get('records', []):
        record_to_node.setdefault(str(rec['recordHash']), nh)
    for child in children.get('presentationNodes', []):
        node_parent.setdefault(str(child['presentationNodeHash']), nh)

def node_name(nh):
    return (nodes_raw.get(str(nh), {}).get('displayProperties', {}).get('name') or '').strip()

lore_to_book = {}
for rh, rec in records_raw.items():
    lh = rec.get('loreHash')
    if not lh:
        continue
    nh = record_to_node.get(rh)
    if nh is None:
        parents = rec.get('presentationInfo', {}).get('parentPresentationNodeHashes', [])
        nh = str(parents[0]) if parents else None
    if nh:
        lore_to_book.setdefault(str(lh), (node_name(nh), node_name(node_parent.get(nh, ''))))

print(f'Lore entries with a book: {len(lore_to_book):,}')

Lore entries with a book: 1,085


In [4]:
def build_d2_docs(lore_raw, lore_to_book):
    rows, dropped = [], {'empty': 0, 'redacted': 0, 'too_short': 0}
    for h, entry in lore_raw.items():
        props = entry.get('displayProperties', {})
        text = (props.get('description') or '').strip()
        if not text:
            dropped['empty'] += 1; continue
        if entry.get('redacted'):
            dropped['redacted'] += 1; continue
        if len(text.split()) < MIN_WORDS:
            dropped['too_short'] += 1; continue
        book, shelf = lore_to_book.get(str(h), ('', ''))
        rows.append({'hash': str(h), 'title': str(props.get('name') or '').strip(),
                     'subtitle': (entry.get('subtitle') or '').strip(),
                     'text': text, 'book': book, 'shelf': shelf})
    df = pd.DataFrame(rows)
    # Dedup: the same text is often attached to several items. Keep the copy that has a book.
    df['nh'] = df['text'].map(norm_hash)
    df['has_book'] = df['book'] != ''
    df = df.sort_values(['nh', 'has_book', 'hash'], ascending=[True, False, True])
    dup_hashes = df.groupby('nh')['hash'].apply(lambda s: list(s)[1:])
    before = len(df)
    df = df.drop_duplicates('nh').copy()
    df['duplicate_hashes'] = df['nh'].map(dup_hashes)
    dropped['duplicate'] = before - len(df)

    docs = []
    for r in df.itertuples():
        docs.append({
            'doc_id': f'd2-{r.hash}', 'title': r.title, 'subtitle': r.subtitle, 'text': r.text,
            'source': 'bungie_api_d2', 'format': 'json', 'game': 'D2',
            'book': r.book, 'shelf': r.shelf, 'role': 'lore',
            'url': f'{BUNGIE}/Platform/Destiny2/Manifest/DestinyLoreDefinition/{r.hash}/',
            'duplicate_ids': [f'd2-{x}' for x in r.duplicate_hashes],
        })
    return docs, dropped

d2_docs, d2_dropped = build_d2_docs(lore_raw, lore_to_book)
print(f'D2 documents: {len(d2_docs):,} | dropped: {d2_dropped}')

D2 documents: 2,526 | dropped: {'empty': 552, 'redacted': 15, 'too_short': 27, 'duplicate': 394}


## 2. Destiny 1 Grimoire (CSV, text contains HTML)

The card text has HTML entities (`&#39;`, `&amp;`) and `<br/>` tags. They are unescaped and stripped here, before anything is embedded.

In [5]:
def clean_grimoire_text(t):
    t = html.unescape(str(t))
    t = re.sub(r'<br\s*/?>', '\n', t, flags=re.I)
    t = BeautifulSoup(t, 'html.parser').get_text()
    t = html.unescape(t)
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n\s*\n+', '\n\n', t)
    return t.strip()

def build_d1_docs(csv_path):
    g = pd.read_csv(csv_path)
    dropped = {'too_short': 0, 'duplicate': 0}
    seen, docs = set(), []
    for i, r in g.iterrows():
        text = clean_grimoire_text(r['text'])
        if len(text.split()) < MIN_WORDS:
            dropped['too_short'] += 1; continue
        h = norm_hash(text)
        if h in seen:
            dropped['duplicate'] += 1; continue
        seen.add(h)
        name = html.unescape(str(r['name'])).strip()
        docs.append({
            'doc_id': f'd1-{i:04d}-{slug(name)}', 'title': name, 'subtitle': '', 'text': text,
            'source': 'bungie_api_d1_grimoire', 'format': 'csv', 'game': 'D1',
            'book': html.unescape(str(r['page'])).strip(), 'shelf': html.unescape(str(r['theme'])).strip(),
            'role': 'lore', 'url': 'https://www.bungie.net/d1/Platform/Destiny/Vanguard/Grimoire/Definition/',
            'duplicate_ids': [],
        })
    return docs, dropped

d1_docs, d1_dropped = build_d1_docs(RAW / 'destiny1_grimoire.csv')
print(f'D1 documents: {len(d1_docs):,} | dropped: {d1_dropped}')

D1 documents: 796 | dropped: {'too_short': 26, 'duplicate': 0}


/Users/henryhalvorsen/anaconda3/lib/python3.10/site-packages/bs4/__init__.py:435: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  warnings.warn(


## 3. Destiny Wiki pages (HTML)

Pages come from the wiki's MediaWiki API (`action=parse`), not by scraping the site. Two lists:
- **Lore pages:** major characters, factions, places, and events
- **Distractor pages:** gameplay and mechanics pages. They use the same vocabulary (raids, Light, Exotic) but don't answer lore questions, so they test whether retrieval picks the right document.

Titles that don't exist on the wiki are logged and skipped. Edit the lists freely; the raw HTML of every page is cached in `data/corpus/raw/wiki/`.

In [6]:
WIKI_API = 'https://destiny.fandom.com/api.php'

WIKI_LORE_PAGES = [
    # Characters
    'Oryx', 'Savathûn', 'Xivu Arath', 'Crota', 'The Witness', 'Cayde-6', 'Zavala', 'Ikora Rey',
    'Eris Morn', 'Osiris', 'Saint-14', 'Mara Sov', 'Uldren Sov', 'Rasputin', 'Dominus Ghaul',
    'Calus', 'Caiatl', 'Mithrax', 'Eramis', 'Variks', 'Toland, the Shattered', 'Elsie Bray',
    'Clovis Bray I', 'Ana Bray', 'Shaxx', 'Riven', 'Quria', 'Nezarec', 'Shin Malphur', 'Dredgen Yor',
    # Peoples and factions
    'Hive', 'Eliksni', 'Cabal', 'Vex', 'Taken', 'Scorn', 'Awoken', 'Exo', 'Guardian', 'Ghost',
    'Iron Lords', 'Vanguard',
    # Concepts, places, events
    'Traveler', 'Darkness', 'Light', 'Golden Age', 'Collapse', 'Red War', 'Taken War',
    'Battle of Twilight Gap', 'Books of Sorrow', 'Worm gods', 'Ahamkara', 'The Nine', 'Pyramid',
    'Dreadnaught', 'Last City', 'Dreaming City', 'Sword logic', 'SIVA', 'Black Garden',
]

WIKI_DISTRACTOR_PAGES = [
    'Hand Cannon', 'Auto Rifle', 'Pulse Rifle', 'Scout Rifle', 'Sniper Rifle', 'Shotgun', 'Fusion Rifle',
    'Rocket Launcher', 'Grenade Launcher', 'Trace Rifle', 'Combat Bow', 'Glaive', 'Machine Gun',
    'Power Level', 'Exotic', 'Legendary', 'Masterwork', 'Mod', 'Weapon Perk', 'Fireteam', 'Clan',
    'Glimmer', 'Bright Dust', 'Silver', 'Eververse', 'Engram', 'Seasonal Artifact', 'Triumph',
    'Seal', 'Transmog', 'Finisher', 'Emote', 'Patrol', 'Public Event', 'Lost Sector', 'Bounty',
    'Nightfall', 'Strike', 'Dungeon', 'Raid', 'Crucible', 'Trials of Osiris', 'Iron Banner', 'Gambit',
]

session = requests.Session()
session.headers['User-Agent'] = USER_AGENT

def fetch_wiki_html(title):
    # Returns (canonical title, html), using the cache when possible. None if the page doesn't exist.
    cache = WIKI_RAW / f'{slug(title)}.json'
    if cache.exists():
        d = json.loads(cache.read_text())
    else:
        r = session.get(WIKI_API, params={'action': 'parse', 'page': title, 'prop': 'text',
                                           'redirects': 1, 'format': 'json', 'formatversion': 2},
                        timeout=30)
        r.raise_for_status()
        d = r.json()
        time.sleep(WIKI_SLEEP)
        if 'error' in d:
            return None
        cache.write_text(json.dumps(d))
    if 'parse' not in d:
        return None
    return d['parse']['title'], d['parse']['text']

CUT_SECTIONS = {'references', 'notes', 'gallery', 'list of appearances', 'appearances', 'sources',
                'external links', 'see also', 'navigation'}
DROP_SELECTORS = ['aside', 'table', 'style', 'script', 'figure', 'sup', '.navbox', '.toc', '#toc',
                  '.mw-editsection', '.reference', '.references', '.wikia-gallery', '.gallery',
                  '.thumb', '.portable-infobox', '.noprint', '.mw-empty-elt']

def clean_wiki_html(page_html):
    soup = BeautifulSoup(page_html, 'html.parser')
    for sel in DROP_SELECTORS:
        for el in soup.select(sel):
            el.decompose()
    parts = []
    for el in soup.find_all(['h2', 'h3', 'h4', 'p', 'li', 'blockquote', 'dd']):
        if el.find_parent(['li', 'blockquote', 'dd']):
            continue  # its text is already included through the parent
        text = re.sub(r'\s+', ' ', el.get_text(' ')).strip()
        if not text:
            continue
        if el.name in ('h2', 'h3', 'h4'):
            if text.lower() in CUT_SECTIONS:
                if el.name == 'h2':
                    break  # everything after a References/Gallery heading is back matter
                continue
            parts.append(f'\n## {text}')
        else:
            parts.append(f'- {text}' if el.name == 'li' else text)
    text = '\n'.join(parts).strip()
    return re.sub(r'\s+([.,;:!?])', r'\1', text)

def build_wiki_docs(titles, role):
    docs, missing = [], []
    for t in titles:
        got = fetch_wiki_html(t)
        if got is None:
            missing.append(t); continue
        canon, page_html = got
        text = clean_wiki_html(page_html)
        if len(text.split()) < MIN_WORDS:
            missing.append(f'{t} (too short after cleaning)'); continue
        docs.append({
            'doc_id': f'wiki-{slug(canon)}', 'title': canon, 'subtitle': '', 'text': text,
            'source': 'destiny_fandom_wiki', 'format': 'html', 'game': 'D1/D2',
            'book': '', 'shelf': 'wiki', 'role': role,
            'url': 'https://destiny.fandom.com/wiki/' + canon.replace(' ', '_'),
            'duplicate_ids': [],
        })
    return docs, missing

wiki_lore, miss_lore = build_wiki_docs(WIKI_LORE_PAGES, 'lore')
wiki_distractors, miss_dist = build_wiki_docs(WIKI_DISTRACTOR_PAGES, 'distractor')
# A redirect can send two titles to the same page; keep one
_seen = set()
wiki_docs = [d for d in wiki_lore + wiki_distractors if not (d['doc_id'] in _seen or _seen.add(d['doc_id']))]
print(f'Wiki lore pages: {len(wiki_lore)} | distractor pages: {len(wiki_distractors)}')
print('Missing or skipped:', miss_lore + miss_dist or 'none')

Wiki lore pages: 55 | distractor pages: 35
Missing or skipped: ['Dominus Ghaul', 'Nezarec (too short after cleaning)', 'Worm gods', 'Pyramid', 'Last City', 'Sword logic', 'Combat Bow', 'Power Level', 'Mod', 'Bright Dust', 'Seasonal Artifact', 'Triumph', 'Seal', 'Transmog', 'Finisher']


## 4. Write the corpus, manifest, and report

In [7]:
LICENSES = {
    'bungie_api_d2': 'Copyright Bungie, Inc. Retrieved through the Bungie.net API under its Terms of Use. '
                     'Text is not redistributed; this notebook re-fetches it.',
    'bungie_api_d1_grimoire': 'Copyright Bungie, Inc. Retrieved through the Bungie.net Destiny 1 API under its Terms of Use. '
                              'Text is not redistributed.',
    'destiny_fandom_wiki': 'CC BY-SA 3.0, Destiny Wiki (destiny.fandom.com) contributors. Attribution: page URL.',
}

all_docs = d2_docs + d1_docs + wiki_docs

# Cross-source dedup: drop any document whose text exactly matches one already kept
seen, corpus, cross_dupes = set(), [], 0
for d in all_docs:
    h = norm_hash(d['text'])
    if h in seen:
        cross_dupes += 1; continue
    seen.add(h)
    d['words'] = len(d['text'].split())
    d['tokens'] = n_tokens(d['text'])
    d['sha1'] = hashlib.sha1(d['text'].encode()).hexdigest()
    d['license'] = LICENSES[d['source']]
    corpus.append(d)

with open(DOCS_PATH, 'w') as f:
    for d in corpus:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

MANIFEST_FIELDS = ['doc_id', 'title', 'source', 'format', 'game', 'book', 'shelf', 'role',
                   'url', 'license', 'words', 'tokens', 'sha1', 'duplicate_ids']
with open(META_DIR / 'manifest.jsonl', 'w') as f:
    for d in corpus:
        f.write(json.dumps({k: d[k] for k in MANIFEST_FIELDS}, ensure_ascii=False) + '\n')

print(f'Wrote {len(corpus):,} documents ({cross_dupes} cross-source duplicates dropped)')

Wrote 3,411 documents (0 cross-source duplicates dropped)


In [8]:
m = pd.DataFrame(corpus)
total_tokens = int(m['tokens'].sum())
n_sources, n_formats = m['source'].nunique(), m['format'].nunique()
distractor_share = (m['role'] == 'distractor').mean()
distractor_token_share = m.loc[m['role'] == 'distractor', 'tokens'].sum() / total_tokens

checks = [
    ('Documents after dedup', '50+', f'{len(m):,}', len(m) >= 50),
    ('Tokens', 'about 250k', f'{total_tokens:,}', total_tokens >= 200_000),
    ('Sources', '3+', str(n_sources), n_sources >= 3),
    ('Formats', '2+', f"{n_formats} ({', '.join(sorted(m['format'].unique()))})", n_formats >= 2),
    ('Distractors (share of documents)', 'about 15%', f'{distractor_share:.1%}', 0.10 <= distractor_share <= 0.20),
    ('Distractors (share of tokens)', 'about 15%', f'{distractor_token_share:.1%}', 0.10 <= distractor_token_share <= 0.20),
    ('License note per document', 'all', f"{m['license'].notna().mean():.0%}", m['license'].notna().all()),
]

by_source = (m.groupby(['source', 'format', 'role'])
               .agg(documents=('doc_id', 'count'), tokens=('tokens', 'sum'),
                    median_words=('words', 'median'))
               .reset_index())

lines = ['# Corpus report', '',
         f"Built {time.strftime('%Y-%m-%d')} from Bungie manifest version {MANIFEST_VERSION}.", '',
         '| Target | Goal | Actual | Met |', '|---|---|---|---|']
lines += [f'| {a} | {b} | {c} | {"yes" if ok else "no"} |' for a, b, c, ok in checks]
src_table = ['| ' + ' | '.join(by_source.columns) + ' |', '|' + '---|' * len(by_source.columns)]
src_table += ['| ' + ' | '.join(f'{v:,.0f}' if isinstance(v, numbers.Number) else str(v) for v in row) + ' |'
              for row in by_source.itertuples(index=False)]
lines += ['', '## By source', ''] + src_table + ['',
          '## Dropped during cleaning', '',
          f'- Destiny 2: {d2_dropped}', f'- Destiny 1: {d1_dropped}',
          f'- Cross-source exact duplicates: {cross_dupes}',
          f'- Wiki titles missing or too short: {miss_lore + miss_dist or "none"}']
report = '\n'.join(lines)
(META_DIR / 'corpus_report.md').write_text(report + '\n')
print(report)

# Corpus report

Built 2026-10-24 from Bungie manifest version 245114.26.10.13.2000-1-bnet.66137.

| Target | Goal | Actual | Met |
|---|---|---|---|
| Documents after dedup | 50+ | 3,411 | yes |
| Tokens | about 250k | 1,318,375 | yes |
| Sources | 3+ | 3 | yes |
| Formats | 2+ | 3 (csv, html, json) | yes |
| Distractors (share of documents) | about 15% | 1.0% | no |
| Distractors (share of tokens) | about 15% | 1.2% | no |
| License note per document | all | 100% | yes |

## By source

| source | format | role | documents | tokens | median_words |
|---|---|---|---|---|---|
| bungie_api_d1_grimoire | csv | lore | 796 | 154,830 | 82 |
| bungie_api_d2 | json | lore | 2,526 | 1,071,072 | 266 |
| destiny_fandom_wiki | html | distractor | 34 | 15,863 | 202 |
| destiny_fandom_wiki | html | lore | 55 | 76,610 | 367 |

## Dropped during cleaning

- Destiny 2: {'empty': 552, 'redacted': 15, 'too_short': 27, 'duplicate': 394}
- Destiny 1: {'too_short': 26, 'duplicate': 0}
- Cross-source exact d

## 5. Quick look

A few random documents from each source, to check the cleaning by eye.

In [9]:
for src in m['source'].unique():
    sample = m[m['source'] == src].sample(1, random_state=0).iloc[0]
    print(f"=== {src} | {sample['doc_id']} | {sample['title']} | book: {sample['book'] or '-'}")
    print(sample['text'][:600], '\n')

=== bungie_api_d2 | d2-317465074 | Cerulean Flash | book: -
Another day of gray and cold. Emmerza had already cried twice this morning and the shop wasn't even open yet. She looks up past the three-day-old pears and pauses. Is that…

WHOOSH! A white blur. The windows rattle. An apple falls to the floor with a thump. And the smile doesn't leave her face even after the fruit's all stocked.

Mist-1 had been looking forward to the air show for months. The forecast said today was finally going to be clear. But since this morning, everything above the sketches on their locker is fog. They sigh. After missing last year's with their sickness, for this year's t 

=== bungie_api_d1_grimoire | d1-0518-xxxix-open-your-eye-go-into-it | XXXIX: open your eye : go into it | book: Books of Sorrow
The Vex clattered around, constructing large problems. At first their constructions were deranged, because they didn’t understand the sword logic, which defined all rules in Oryx’s throne world. The geometry p